Задание 4
Вам предоставлены данные о людях с наличием или отсутствием заболевания:

Скачать файл с данными

Признаки:

Age — возрастная группа.
Test — позитивный или негативный тест на заболевание.
Status — целевая переменная, есть инфекция или нет.
Необходимо реализовать алгоритм Наивного Байеса для решения задачи классификации. Обязательно оцените качество полученного результата по итогу.

In [37]:

import numpy as np
import pandas as pd

med_data = pd.read_csv('C:/IDE/Math/exam/asset-v1_SkillFactory+MIFIML-1sem+2024+type@asset+block@sf_exam.csv')

train_data = med_data.sample(frac=0.8, random_state=42)
test_data = med_data.drop(train_data.index)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

# Вероятность инфекции
P_infected = train_data['Status'].value_counts(normalize=True)
# Общее количество
N_status = train_data['Status'].value_counts()

# Условные вероятности(сглаживание Лапласа)
alpha = 1 

# P(Age_Group | Status)
P_age_status = (train_data.groupby(['Status', 'Age_Group']).size() + alpha) / (train_data.groupby('Status').size() + train_data['Age_Group'].nunique() * alpha)

#  P(Test_Result | Status)
P_test_status = (train_data.groupby(['Status', 'Test']).size() + alpha) / (train_data.groupby('Status').size() + train_data['Test'].nunique() * alpha)


# Наивный Байес с лог вер и сглаж Лапласа
def predict_nb(row):
    probs = {}
    for status in N_status.index:
        # Лог апр вер
        log_p_status = np.log(N_status[status] / N_status.sum())
        # Лог усл вер признаков
        log_p_age = np.log(P_age_status.get((status, row['Age_Group']), 
                                            alpha / (N_status[status] + train_data['Age_Group'].nunique() * alpha)))
        log_p_test = np.log(P_test_status.get((status, row['Test']), 
                                              alpha / (N_status[status] + train_data['Test'].nunique() * alpha)))
        # Сумма
        probs[status] = log_p_status + log_p_age + log_p_test

    return max(probs, key=probs.get)

# Применяем к тестовым данным
test_data['Predicted_Status'] = test_data.apply(predict_nb, axis=1)

# Качество модели
accuracy = np.mean(test_data['Predicted_Status'] == test_data['Status'])
print(f"Точность модели: {accuracy:.2%}")

# Матрица ошибок
conf_matrix = pd.crosstab(test_data['Status'], 
                          test_data['Predicted_Status'], 
                          rownames=['Факт'], 
                          colnames=['Прогноз'])
print("\nМатрица ошибок:")
print(conf_matrix)



print(f"\nTP: {conf_matrix.loc['Infected', 'Infected']}")
print(f"FN: {conf_matrix.loc['Infected', 'Not_infected']}")
print(f"FP: {conf_matrix.loc['Not_infected', 'Infected']}")
print(f"TN: {conf_matrix.loc['Not_infected', 'Not_infected']}")




Точность модели: 83.93%

Матрица ошибок:
Прогноз       Infected  Not_infected
Факт                                
Infected            22             6
Not_infected         3            25

TP: 22
FN: 6
FP: 3
TN: 25


Задание 5. Дана функция: f(x) = x**4 + 3*x**3 - 12*x**2 + 7*x -2
Оптимизируйте её с помощью метода Ньютона, используя язык программирования Python.

За начальную точку возьмите х = 10, необходимую точность 0.0001.

In [38]:
def f_prime(x):
    return 4*x**3 + 9*x**2 - 24*x + 7

def f_double_prime(x):
    return 12*x**2 + 18*x - 24

def newton_optimize(x_0, eps=1e-4, max_iter=1000):
    x = x_0
    for i in range(max_iter):
        f_p = f_prime(x)
        f_pp = f_double_prime(x)
        
        print(f"Итерация {i}: x = {x:.6f}, f'(x) = {f_p:.8f}")
        
        # близость производной к нулю
        if abs(f_p) < eps:
            print(f"Решение найдено за {i+1} итераций. |f'(x)| < {eps}")
            return x
        
        if abs(f_pp) < 1e-12:
            print("Вторая производная близка к нулю, остановка.")
            return x
        
        x_new = x - f_p / f_pp
        x = x_new
    
    print("Достигнут максимум итераций.")
    return x


x_0 = 10
result = newton_optimize(x_0)
print(f"\nНайденный локальный экстремум: x = {result:.8f}")
print(f"Значение f'(x) в этой точке: {f_prime(result):.10f}")
print(f"Проверка |f'(x)| < 0.0001: {abs(f_prime(result)) < 0.0001}")

Итерация 0: x = 10.000000, f'(x) = 4667.00000000
Итерация 1: x = 6.558260, f'(x) = 1365.00183680
Итерация 2: x = 4.321204, f'(x) = 394.10233966
Итерация 3: x = 2.902832, f'(x) = 111.01197067
Итерация 4: x = 2.044723, f'(x) = 29.74971911
Итерация 5: x = 1.572323, f'(x) = 7.06241661
Итерация 6: x = 1.364410, f'(x) = 1.16871305
Итерация 7: x = 1.313372, f'(x) = 0.06556228
Итерация 8: x = 1.310148, f'(x) = 0.00025712
Итерация 9: x = 1.310136, f'(x) = 0.00000000
Решение найдено за 10 итераций. |f'(x)| < 0.0001

Найденный локальный экстремум: x = 1.31013560
Значение f'(x) в этой точке: 0.0000000040
Проверка |f'(x)| < 0.0001: True


Задание 6
Дано распределение случайной величины, которая отражает вероятность получения разного количества спам-писем в течение дня.
1. Найдите вероятность получения 7 спам-писем.
2. Найдите математическое ожидание для количества полученных писем.
3. Найдите дисперсию для количества полученных писем. Ответ округлите до сотых.
Задание должно быть решено без использования готовых функций Python.

In [39]:

probabilities = [0.1, 0.3, 0.1, 0.25, 0.05, 0.05, 0.1] 
x = list(range(8)) 

# 1. Находим вероятность получения 7 спам писем, из 1.0 вычитаем сумму вероятностей получения 0,1,2,3,4,5,6 писем.
sum_known = sum(probabilities)
Prob_7 = 1.0 - sum_known
Prob_7 = round(Prob_7,2)
probabilities.append(Prob_7)

print(f"1. Вероятность получения 7 спам-писем: {Prob_7}")

# 2. Мат ожидание
EX = 0.0
for i in range(8):
    EX += x[i] * probabilities[i]

print(f"\n2. Математическое ожидание: {EX}")

# 3. Дисперсия
#  E[X**2]
EX2 = 0.0
for i in range(8):
    EX2 += (x[i] ** 2) * probabilities[i]

# Дисперсия = E[X**2] - (E[X])**2
DX = EX2 - (EX ** 2)
DX_rounded = round(DX, 2)

print(f"\n3. Дисперсия: {DX_rounded}")


1. Вероятность получения 7 спам-писем: 0.05

2. Математическое ожидание: 2.65

3. Дисперсия: 4.03


Задание 7
За 8 часов рабочего дня в колл-центр поступает в среднем 16 звонков. Найдите вероятность, что за час в колл-центр поступит не более 4 и не менее 2 звонков. Ответ округлите до тысячных. Задание должно быть решено без использования готовых функций Python.

In [42]:
#Мы имеем пуассоновский процесс: в среднем 16 звонков за 8 часов ⇒ λ = 16/8 = 2 звонка в час.
# P(x=2,3,4) для распределения Пуассона с λ = 2.

lam = 2
e_lam = 0.135335283  # e**(-λ)

P2 = (lam**2) * e_lam / 2
P3 = (lam**3) * e_lam / 6
P4 = (lam**4) * e_lam / 24

total = round(P2 + P3 + P4, 3)


print("P(2) =", P2)
print("P(3) =", P3)
print("P(4) =", P4)
print("Сумма =", total)

P(2) = 0.270670566
P(3) = 0.180447044
P(4) = 0.090223522
Сумма = 0.541
